In [1]:
import numpy as np
import pandas as pd
import glob, os, vcf, warnings, shutil, subprocess, re, sys, gzip
from Bio import Seq, SeqIO
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
import scipy, pysam
from matplotlib.colors import ListedColormap
from collections import Counter
from sklearn.preprocessing import StandardScaler
import sklearn.model_selection

df_Illumina_WGS = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/Illumina_culture_WGS_summary.csv")

df_RedCap = pd.read_csv(f"~/TRUST_data_processing/raw_data/TRUST_DATA_2025-05-12_1129.cleaned.wide.csv")

Max_LR_metadata = pd.read_csv("~/TRUST_data_processing/raw_data/230902.TRUST.LRandSR.Set1to4.106CI.InputWGS.PATHs.tsv", sep='\t')

personal_ref_dir = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_personal_assembly"
H37Rv_ref_dir = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_lowAF"
assembly_dir = "/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBio_Illumina_hybridASM"

Max_ASM_QC = pd.read_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/TRUST.PBAsm.ZA_106CI_AssemblySummary.csv")
Max_ASM_QC['Lineage'] = Max_ASM_QC['Lineage_Asm'].str.replace('lineage', '')

# this keeps only those with NumContigs = 1 and numContigs_Complete = 1 in df_LR_assembly_metadata, and the lineages (Coll2014 exactly) of the LR and SR sequences must match
df_personal_assemblies = pd.read_csv("./data/personal_assemblies_samples.tsv", sep='\t', header=None, names = ['Sample', 'Assembly', 'PacBio'])
samples_with_assemblies = df_personal_assemblies['Sample'].values

df_trust_patients = pd.read_csv("~/TRUST_data_processing/processed_data/combined_patient_WGS_data.csv")

F2_max = 0.03

In [3]:
Max_ASM_QC = pd.read_csv("~/210305_PMP_48CI_CircularOnly_F2Filtered_AtLeast40XMeanDepthIllumina_AssemblySummary_V7.tsv", sep='\t')
Max_LR_FASTQ = pd.read_csv("~/220719.Hall2022.80CI.Peker2021.18CI.MtbWGS.LRandSR.CircAsmAndPEIllumina.PATHs.tsv", sep='\t')

In [5]:
# df_samples = pd.read_csv("~/MtbLongitudinalDiversity/lowAF_variant_calling/data/TRUST_SR_samples_personal_genomes.tsv", sep='\t', header=None)
# df_samples.columns = ['SampleID', 'ASM_original', 'ASM_local']
# len(df_samples)

In [6]:
def get_samples_to_align_to_personal_ref_genome(lineage_string):
    
    lineage_lst = lineage_string.split(',')
    
    samples_needed = df_trust_patients.query("Coll2014 in @lineage_lst & SampleID in @unmixed_lineage_samples & SampleID in @samples_with_assemblies").SampleID.unique()
    
    # take the first one as the assembly. When running this on all samples, remove the requirement that SampleID be in samples_with_assemblies when getting samples_needed
    assembly_sample = np.sort(samples_needed)[0]
    
    # then remove assembly_sample from samples_needed
    samples_needed = list(set(samples_needed) - set([assembly_sample]))
    
    for name in samples_needed:
        if not os.path.isfile(f"{H37Rv_ref_dir}/{name}/{name}/kraken/{name}.R1.kraken.filtered.fastq.gz"):
            print(name)
            
    if len(lineage_lst) == 1:
        saveName = f"SR_samples_align_{lineage_lst[0]}_{assembly_sample}_ASM.txt"
    else:
        saveName = f"SR_samples_align_{','.join(lineage_lst)}_{assembly_sample}_ASM.txt"
      
    if len(samples_needed) > 0:
        print(f"{len(samples_needed)} will be aligned to the {assembly_sample} assembly ({lineage_string})")
        pd.Series(samples_needed).to_csv(saveName, sep='\t', header=None, index=False)

In [8]:
# combine_lineage_dict = {'2.2.1,2.2.1.1': False,
#                         '3,3.1.1': False,
#                         '4.1.1.1,4.1.1.3,4.1.2': False
#                        }

# only combine these because there's only 1 of each, so there's no way to know if using another sample works, unless we broaden the search a little bit
combine_lineage_dict = {'3,3.1.1': False}

for lineage in df_trust_patients.dropna(subset='Coll2014').query("SampleID in @samples_with_assemblies & ~Coll2014.str.contains(',')").Coll2014.unique():
    
    combo_lineage = False
    
    for lineage_string in combine_lineage_dict.keys():
        if len(set(lineage_string.split(',')).intersection([lineage])) > 0:
            combo_lineage = True
            break
        
    if combo_lineage:
        if not combine_lineage_dict[lineage_string]:
            get_samples_to_align_to_personal_ref_genome(lineage_string)
            combine_lineage_dict[lineage_string] = True
            # print(f"{lineage} is in a combo")
    else:
        get_samples_to_align_to_personal_ref_genome(lineage)
        # print(lineage)

In [4]:
def compare_variants_two_personal_ref_genomes(assembly_sample, variants_personal_fName, variants_other_personal_fName):

    assert os.path.isfile(variants_personal_fName)
    assert os.path.isfile(variants_other_personal_fName)
        
    # if this fails, it's empty
    try:
        df_variants_personal = pd.read_csv(variants_personal_fName, sep='\t', header='infer')#, usecols=[0, 1, 2])
        # df_variants_personal.columns = ['CHROM', 'BEG', 'END']
        df_variants_personal[['BEG', 'END']] = df_variants_personal[['BEG', 'END']].astype(int)
    except:
        df_variants_personal = pd.DataFrame(columns = ['CHROM', 'BEG', 'END'], index=[0])
        
    try:
        df_variants_other_personal = pd.read_csv(variants_other_personal_fName, sep='\t', header=None, usecols=[0, 1, 2, 3])
        df_variants_other_personal.columns = ['ASM_CHROM', 'ASM_BEG', 'ASM_END', 'SplitCol']
        df_variants_other_personal[['CHROM', 'BEG', 'END']] = df_variants_other_personal['SplitCol'].str.rsplit('_', n=2, expand=True)
        df_variants_other_personal[['ASM_BEG', 'ASM_END']] = df_variants_other_personal[['ASM_BEG', 'ASM_END']].astype(int)

        # to merge with the TSV of variant information. This file is just the BED file, so it doesn't have variant information
        df_variants_other_personal['POS'] = df_variants_other_personal['BEG'].astype(int) + 1 
    except:
        df_variants_other_personal = pd.DataFrame(columns = ['ASM_CHROM', 'ASM_BEG', 'ASM_END', 'POS'], index=[0])

    # get the variant information
    other_ASM_variants_fName = os.path.join(os.path.dirname(variants_other_personal_fName), f"{sample}.excludeLowConf.tsv")
    df_other_ASM_variants = pd.read_csv(other_ASM_variants_fName, sep='\t')
    df_other_ASM_variants['AF'] = df_other_ASM_variants['AO'] / df_other_ASM_variants['DP']

    df_variants_other_personal = df_variants_other_personal.merge(df_other_ASM_variants[['POS', 'REF', 'ALT', 'AF']], on='POS')

    # remove variants at high frequency. These may not be picked up because slight differences can cause one variant to be picked up at AF = 0.04 vs. AF = 0.06
    # then it'll come up as a false negative or positive because of the threshold we used, requiring that a low frequency variant be ≥ 0.05
    df_variants_other_personal = df_variants_other_personal.query("AF <= 0.95")

    del_cols = []

    # so that after merging, we don't get multiples. These are not always going to match
    for col in ['CHROM', 'BEG', 'END', 'POS', 'REF', 'ALT', 'AF']:
        if col in df_variants_other_personal.columns:
            del df_variants_other_personal[col]

    df_variants_comparison = df_variants_personal.merge(df_variants_other_personal, 
                                                        left_on=['CHROM', 'BEG', 'END'], 
                                                        right_on=['ASM_CHROM', 'ASM_BEG', 'ASM_END'], 
                                                        how='outer')

    df_variants_comparison['ASM'] = assembly_sample

    # ONLY SNPs
    df_variants_comparison['varLen'] = df_variants_comparison['END'] - df_variants_comparison['BEG']
    df_variants_comparison['ASM_varLen'] = df_variants_comparison['ASM_END'] - df_variants_comparison['ASM_BEG']

    # have to do ~varLen > 1 so that you don't drop cases that are NA. NAs are those found in one BED file but not in another (important for tabulating results!!!)
    df_variants_comparison = df_variants_comparison.loc[~(df_variants_comparison["varLen"] > 1) & ~(df_variants_comparison["ASM_varLen"] > 1)].reset_index(drop=True)

    if len(df_variants_comparison) > 0:

        # found in personal ref genome, but not using the other personal ref genome. Means missed = FN
        df_variants_comparison.loc[(~pd.isnull(df_variants_comparison['BEG'])) & (pd.isnull(df_variants_comparison['ASM_BEG'])), 'Result'] = 'FN'

        # not found in personal ref genome, but found using the other personal ref genome. Means FP
        df_variants_comparison.loc[(pd.isnull(df_variants_comparison['BEG'])) & (~pd.isnull(df_variants_comparison['ASM_BEG'])), 'Result'] = 'FP'

        # found in both. Means correct = TP
        df_variants_comparison.loc[(~pd.isnull(df_variants_comparison['BEG'])) & (~pd.isnull(df_variants_comparison['ASM_BEG'])), 'Result'] = 'TP'

        return df_variants_comparison.dropna(axis=1, how='all')

# Matching Illumina to PacBio Samples

In [9]:
# Pacbio_updated_table = pd.read_excel("~/TRUST_data_processing/raw_data/PacBio_update2025.xlsx")

Max_LR_metadata = pd.read_csv("~/TRUST_data_processing/raw_data/230902.TRUST.LRandSR.Set1to4.106CI.InputWGS.PATHs.tsv", sep='\t')

Max_LR_metadata = Max_LR_metadata.rename(columns={'SampleID': 'Original_ID'}).merge(df_trust_patients[['Original_ID', 'SampleID']])

df_LR_SR = pd.read_csv("/home/sak0914/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data.tsv", sep='\t')

len(df_LR_SR), len(Max_LR_metadata), len(set(Max_LR_metadata.Original_ID) - set(df_LR_SR.Original_ID))

(193, 106, 0)

In [11]:
def read_in_PacBio_samples(fName):
    
    try:
        df = pd.read_csv(fName, usecols=[0,1]).dropna(how='all', axis=1)
        
        if df.shape[1] == 1:
            if ';' in df.iloc[:, 0].values[0]:
                df_expanded = df.iloc[:, 0].str.split(';', expand=True)
                df = df_expanded[[0, 1]]
        
        df.columns = ['Sample', 'Barcode']
        
    except:
        df = pd.read_csv(fName).reset_index().iloc[:, :2]
    
        df.columns = ['Sample', 'Barcode']
        
        if ';' in df.Sample.values[0]:
            df_expanded = df['Sample'].str.split(';', expand=True)
            df = df_expanded[[0, 1]]
    
    # add the directory name to get the FASTQ files
    df['Directory'] = os.path.dirname(fName)
    df['FQ_prefix'] = os.path.basename(fName).replace('.barcodes_summary.csv', '')
    
    return df

In [12]:
PacBio_tables = glob.glob("/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/*/*.csv")
print(len(PacBio_tables))

PacBio_dirs = np.unique([os.path.dirname(fName) for fName in PacBio_tables])
print(len(PacBio_dirs))

8
7


In [13]:
df_PacBio_metadata = []

for fName in PacBio_tables:
    df = read_in_PacBio_samples(fName)
    df_PacBio_metadata.append(df)

df_PacBio_metadata = pd.concat(df_PacBio_metadata).reset_index(drop=True)

print(df_PacBio_metadata.query("~Sample.str.startswith('MFS')"))

df_PacBio_metadata = df_PacBio_metadata.query("Sample.str.startswith('MFS')").reset_index(drop=True)

assert len(set(PacBio_dirs) - set(df_PacBio_metadata.Directory)) == 0

df_PacBio_metadata['FQ_prefix'] = df_PacBio_metadata['FQ_prefix'].replace('TRUST.PB.Batch2', 'demultiplex')
df_PacBio_metadata['PacBio_FQ_PATH'] = df_PacBio_metadata['Directory'] + '/' + df_PacBio_metadata['FQ_prefix'] + '.' + df_PacBio_metadata['Barcode'] + '.hifi_reads.fastq.gz'

for i, row in df_PacBio_metadata.iterrows():
    fName = row['PacBio_FQ_PATH']
    
    if not os.path.isfile(fName):
        df_PacBio_metadata.loc[i, 'PacBio_FQ_PATH'] = fName.replace('hifi_reads', 'ccs')
        assert os.path.isfile(fName.replace('hifi_reads', 'ccs'))
        
df_PacBio_metadata['PacBio_FQ_Name_Only'] = [os.path.basename(fName) for fName in df_PacBio_metadata['PacBio_FQ_PATH'].values]
        
# check no duplicate FQs
print("\n")
print(len(df_PacBio_metadata), df_PacBio_metadata.PacBio_FQ_PATH.nunique(), df_PacBio_metadata.Sample.nunique())

df_PacBio_metadata['#sample'] = df_PacBio_metadata['PacBio_FQ_Name_Only'].str.replace('.fastq.gz', '')

      Sample       Barcode                                          Directory  \
48   No Name  Not Barcoded  /n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...   
80   No Name  Not Barcoded  /n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...   
112  No Name  Not Barcoded  /n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...   
141  No Name  Not Barcoded  /n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...   

                FQ_prefix  
48        TRUST.PB.Batch2  
80   m64142_221216_110244  
112  m64142_221219_140554  
141  m64142_230928_115158  


193 193 193


In [92]:
missing_PacBio_samples = df_PacBio_metadata.query("Sample not in @df_LR_SR.PacBio_ID").Sample.values

df_add = df_PacBio_metadata.query("Sample not in @df_LR_SR.PacBio_ID").merge(df_combined_reports.query("SampleID in @missing_PacBio_samples")[['SampleID', 'original ID']].rename(columns={'SampleID': 'Sample', 'original ID': 'Original_ID'}))

In [93]:
df_add['Dataset_Tag'] = 'TRUST_PB_Set1'
df_add.rename(columns={'Sample': 'PacBio_ID'}, inplace=True)
df_add['Illumina_ID'] = df_add['PacBio_ID']

df_add

,PacBio_ID,Barcode,Directory,FQ_prefix,PacBio_FQ_PATH,PacBio_FQ_Name_Only,#sample,Original_ID,Dataset_Tag,Illumina_ID
0,MFS-43,bc2003,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST.bc2003.ccs.fastq.gz,TRUST.bc2003.ccs,77-01,TRUST_PB_Set1,MFS-43
1,MFS-44,bc2004,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST.bc2004.ccs.fastq.gz,TRUST.bc2004.ccs,77-06,TRUST_PB_Set1,MFS-44
2,MFS-47,bc2005,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST.bc2005.ccs.fastq.gz,TRUST.bc2005.ccs,82-01,TRUST_PB_Set1,MFS-47
3,MFS-136,bc2016,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST.bc2016.ccs.fastq.gz,TRUST.bc2016.ccs,227-07,TRUST_PB_Set1,MFS-136


In [94]:
df_add = df_add.merge(df_Illumina_WGS[['SampleID', 'Kraken_Unclassified_Percent', 'F2', 'Coll2014']], left_on='PacBio_ID', right_on='SampleID')

In [99]:
# del df_LR_SR['Illumina_PE_FQs_PATH']
df_LR_SR = pd.concat([df_LR_SR, df_add[df_LR_SR.columns]])

In [106]:
fastlin_output = []
fastlin_files = glob.glob("/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/*/fastlin_output.txt")

for fName in fastlin_files:
    fastlin_output.append(pd.read_csv(fName, sep='\t'))
    
fastlin_output = pd.concat(fastlin_output)

In [138]:
# df_LR_SR['#sample'] = df_LR_SR['PacBio_FQ_PATH'].transform(lambda x: os.path.basename(x).split('.fastq.gz')[0])
# df_LR_SR = df_LR_SR.merge(fastlin_output[['#sample', 'mixture', 'lineages']], on='#sample')

# del df_LR_SR['SampleID']
# df_LR_SR.rename(columns={'lineages': 'fastlin_output'}, inplace=True)

In [139]:
df_LR_SR['fastlin'] = np.nan

for i, row in df_LR_SR.iterrows():
    
    if not pd.isnull(row['fastlin_output']):
        
        recombine_lineages = []
        
        split_lineages = row['fastlin_output'].split(', ')
        
        for lineage in split_lineages:
            recombine_lineages.append(lineage.split(' (')[0])

        df_LR_SR.loc[i, 'fastlin'] = ','.join(np.sort(recombine_lineages))
        
    if not pd.isnull(row['Coll2014']):
        if ',' in row['Coll2014']:
            df_LR_SR.loc[i, 'Coll2014'] = ','.join(np.sort(row['Coll2014'].split(',')))
            
df_LR_SR.rename(columns={'mixture': 'PacBio_fastlin_mixture', 'fastlin_output': 'PacBio_fastlin_output', 'fastlin': 'PacBio_fastlin'}, inplace=True)

In [147]:
for i, row in df_LR_SR.iterrows():
    
    sample = row['Illumina_ID']
    
    if os.path.isdir(f"/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumina/TRUST.SRWGS.FQs.AllSets.RenamedAndLanesMerged/{sample}"):
        df_LR_SR.loc[i, 'Illumina_FQ_PATH'] = f"/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumina/TRUST.SRWGS.FQs.AllSets.RenamedAndLanesMerged/{sample}"
    else:
        print(sample)

MFS-661
MFS-193


In [162]:
for i, row in df_LR_SR.iterrows():
    
    if not pd.isnull(row['Original_ID']):
        if not row['Original_ID'].startswith('S'):

            length_numerical = len(row['Original_ID'].split('-')[0])

            newID = 'S' + '0' * (4 - length_numerical) + row['Original_ID']
            df_LR_SR.loc[i, 'Original_ID'] = newID

In [166]:
df_LR_SR.to_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data.tsv", sep='\t', index=False)

In [180]:
len(df_LR_SR.dropna(subset='Original_ID')), df_LR_SR.dropna(subset='Original_ID').Original_ID.nunique()

(189, 185)

In [198]:
df_LR_SR.iloc[df_LR_SR.index.values[df_LR_SR.duplicated(subset='Original_ID', keep=False)]].sort_values('Original_ID').dropna(subset='Original_ID')

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Kraken_Unclassified_Percent,F2,Coll2014,#sample,PacBio_fastlin_mixture,PacBio_fastlin_output,PacBio_fastlin,Illumina_FQ_PATH
31,S0077-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set2,MFS-212,MFS-43,0.47,0.021399,4.1.2.1,demultiplex.bc2058--bc2058.hifi_reads,no,4.1.1.3 (74),4.1.1.3,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
189,S0077-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-43,MFS-43,0.47,0.021399,4.1.2.1,TRUST.bc2003.ccs,no,4.1.2.1 (35),4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
95,S0077-06,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set4,MFS-213,MFS-44,0.66,0.021498,4.1.2.1,m64142_221219_140554.bc2080--bc2080.hifi_reads,no,4.1.2.1 (132),4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
190,S0077-06,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-44,MFS-44,0.66,0.021498,4.1.2.1,TRUST.bc2004.ccs,no,4.1.2.1 (22),4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
37,S0082-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set2,MFS-214,MFS-47,0.62,0.012186,3.1.1,demultiplex.bc2064--bc2064.hifi_reads,no,3.1.1 (81),3.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
191,S0082-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-47,MFS-47,0.62,0.012186,3.1.1,TRUST.bc2005.ccs,no,3.1.1 (34),3.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
72,S0227-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set3,MFS-241,MFS-136,0.69,0.007964,4.1.1.1,m64142_221216_110244.bc2087--bc2087.hifi_reads,no,4.1.1.1 (124),4.1.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
192,S0227-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-136,MFS-136,0.69,0.007964,4.1.1.1,TRUST.bc2016.ccs,no,4.1.1.1 (8),4.1.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


In [191]:
df_LR_SR.query("PacBio_ID=='MFS-43'")

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Kraken_Unclassified_Percent,F2,Coll2014,#sample,PacBio_fastlin_mixture,PacBio_fastlin_output,PacBio_fastlin,Illumina_FQ_PATH
189,S0077-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-43,MFS-43,0.47,0.021399,4.1.2.1,TRUST.bc2003.ccs,no,4.1.2.1 (35),4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


In [193]:
set(df_LR_SR.PacBio_ID) - set(df_PacBio_metadata.Sample)

set()

In [194]:
df_PacBio_metadata.query("Sample=='MFS-43'")

,Sample,Barcode,Directory,FQ_prefix,PacBio_FQ_PATH,PacBio_FQ_Name_Only,#sample
2,MFS-43,bc2003,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST.bc2003.ccs.fastq.gz,TRUST.bc2003.ccs


In [187]:
df_LR_SR.query("Original_ID=='S0077-01' & PacBio_ID=='MFS-43'").to_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data_to_run.tsv", sep='\t', index=False)

In [176]:
len(df_LR_SR.query("Original_ID not in @Max_LR_metadata.Original_ID")), len(Max_LR_metadata)

(83, 106)

In [168]:
df_LR_SR.query("Original_ID not in @Max_LR_metadata.Original_ID").to_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data_to_run.tsv", sep='\t', index=False)

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Kraken_Unclassified_Percent,F2,Coll2014,#sample,PacBio_fastlin_mixture,PacBio_fastlin_output,PacBio_fastlin,Illumina_FQ_PATH
106,S0017-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-520,MFS-520,0.42,0.019246,1.1.2,m64142_240409_061126.bc2001--bc2001.hifi_reads,no,1.1.2 (42),1.1.2,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
107,S0063-02,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-550,MFS-550,0.40,0.010625,2.2.1,m64142_240409_061126.bc2002--bc2002.hifi_reads,no,2.2.1 (25),2.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
108,S0097-02,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-566,MFS-566,0.43,0.009978,3,m64142_240409_061126.bc2003--bc2003.hifi_reads,no,3 (59),3,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
109,S0120-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-583,MFS-583,0.89,0.010295,2.2.1,m64142_240409_061126.bc2004--bc2004.hifi_reads,no,2.2.1 (33),2.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
110,S0139-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-601,MFS-601,0.62,0.009904,2.2.1,m64142_240409_061126.bc2005--bc2005.hifi_reads,no,2.2.1 (8),2.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,S0349-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-429,MFS-429,0.71,0.010488,4.1.2.1,m64142_230928_115158.bc2060--bc2060.hifi_reads,no,4.1.2.1 (57),4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
185,S0351-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-430,MFS-430,0.67,0.006222,4.3.3,m64142_230928_115158.bc2061--bc2061.hifi_reads,no,4.3.3 (70),4.3.3,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
186,S0351-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-431,MFS-431,0.72,0.007208,4.3.3,m64142_230928_115158.bc2062--bc2062.hifi_reads,no,4.3.3 (74),4.3.3,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
187,S0353-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-434,MFS-434,0.60,0.009987,2.2.1,m64142_230928_115158.bc2063--bc2063.hifi_reads,no,2.2.1 (106),2.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


In [170]:
set(finished_asms) - set(df_LR_SR.Original_ID)

{'O2logs', 'logs'}

In [169]:
finished_asms = os.listdir(assembly_dir)
len(finished_asms)

187

In [174]:
for fName in finished_asms:
    if fName[0] != 'S':
        print(fName)

O2logs
logs


In [157]:
df_LR_SR.loc[df_LR_SR['#sample']=='m64142_230928_115158.bc2058--bc2058.hifi_reads']

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Kraken_Unclassified_Percent,F2,Coll2014,#sample,PacBio_fastlin_mixture,PacBio_fastlin_output,PacBio_fastlin,Illumina_FQ_PATH
182,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-425,MFS-425,85.23,NaN,NaN,m64142_230928_115158.bc2058--bc2058.hifi_reads,no,NaN,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


In [159]:
fastlin_output.loc[fastlin_output['#sample']=='m64142_240409_061126.bc2013--bc2013.hifi_reads']

,#sample,data_type,k_cov,mixture,lineages,log_barcodes,log_errors
12,m64142_240409_061126.bc2013--bc2013.hifi_reads,single,29,no,NaN,"1.1.2 (20), 1.2.1.2.1 (14)",NaN


In [160]:
fastlin_output

,#sample,data_type,k_cov,mixture,lineages,log_barcodes,log_errors
0,TRUST.bc2001.ccs,single,53,no,2.2.1 (51),"2 (47, 54, 54, 58, 43, 61, 53, 68, 54, 51), 2....",NaN
1,TRUST.bc2002.ccs,single,75,no,2.2.1 (65),"2 (78, 54, 173, 86, 55, 69, 47, 81, 44, 45), 2...",NaN
2,TRUST.bc2003.ccs,single,37,no,4.1.2.1 (35),"4 (29, 50, 42, 22, 47, 22, 34, 32, 36, 32), 4....",NaN
3,TRUST.bc2004.ccs,single,24,no,4.1.2.1 (22),"4 (10, 19, 29, 21, 24, 15, 31, 27, 27, 27), 4....",NaN
4,TRUST.bc2005.ccs,single,44,no,3.1.1 (34),"3 (32, 44, 68, 49, 30, 42, 38, 27, 53), 3.1 (2...",NaN
...,...,...,...,...,...,...,...
37,m64142_250205_154304.bc2040--bc2040.hifi_reads,single,32,no,4.4.1.1.1 (27),"4 (40, 31, 32, 28, 27, 25, 26, 32, 24, 24), 4....",NaN
38,m64142_250205_154304.bc2048--bc2048.hifi_reads,single,31,no,4.1.1.1 (34),"4 (28, 30, 18, 32, 34, 28, 29, 38, 31, 34), 4....",NaN
39,m64142_250205_154304.bc2049--bc2049.hifi_reads,single,37,no,4.3.4.2.1 (39),"4 (41, 32, 30, 35, 40, 29, 25, 43, 39, 42), 4....",NaN
40,m64142_250205_154304.bc2051--bc2051.hifi_reads,single,20,no,2.2.1 (16),"2 (33, 11, 13, 30, 18, 22, 29, 19, 31, 22), 2....",NaN


In [26]:
df_PacBio_metadata.query("Sample not in @df_LR_SR.PacBio_ID").PacBio_FQ_PATH.values

array(['/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2003.ccs.fastq.gz',
       '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2004.ccs.fastq.gz',
       '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2005.ccs.fastq.gz',
       '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2016.ccs.fastq.gz'],
      dtype=object)

In [27]:
df_trust_patients.query("SampleID=='MFS-43'")[['SampleID', 'Coll2014']]

,SampleID,Coll2014
563,MFS-43,4.1.2.1


In [48]:
excel_reports = glob.glob("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/*")
df_combined_reports = []

for fName in excel_reports:
    df_report = pd.read_excel(fName, sheet_name=None)['summary']
    df_combined_reports.append(df_report)
    
df_combined_reports = pd.concat(df_combined_reports).drop_duplicates(subset='SampleID', keep='last')

In [49]:
df_search_PacBio_samples = []
candidate_PacBio_sampleIDs = []

for fName in excel_reports:
    df_report = pd.read_excel(fName, sheet_name=None)
    # print(df_report.keys())
    # print(df_report['statistics']['Source'].unique())
    
    for key in df_report.keys():
        if 'seqorder' in key:
            df_search_PacBio_samples.append(df_report[key])
            print(f"{fName} has seqorder")
            
    if 'Pacbio' in df_report['summary'].columns:
        print(f"{fName} has Pacbio")
        candidate_PacBio_sampleIDs += list(df_report['summary'].query("Pacbio==1").SampleID.values)
    
    # df_combined_reports.append(df_report)
    
candidate_PacBio_sampleIDs = np.unique(candidate_PacBio_sampleIDs)
print(len(df_search_PacBio_samples))

df_search_PacBio_samples = pd.concat(df_search_PacBio_samples)

/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2023-01-24_report_Farhat.v09.xlsx has seqorder
/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2023-09-01_report_Farhat.v09.xlsx has seqorder
/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2024-05-27_report.v10.xlsx has Pacbio
/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2024-06-25_report.v10.xlsx has Pacbio
/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2024-11-13_report.v10.xlsx has Pacbio
/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2025-07-25_report.v10.xlsx has Pacbio
2


In [78]:
len(candidate_PacBio_sampleIDs)

57

In [80]:
Max_LR_metadata.query("SampleID in @candidate_PacBio_sampleIDs")

,Original_ID,PacBio_FQ_PATH,Illumina_PE_FQs_PATH,Dataset_Tag,SampleID


In [158]:
fastlin_output = []

for dir_name in df_PacBio_metadata.query("Sample in @candidate_PacBio_sampleIDs").Directory.unique():
    print(dir_name)
    fastlin_output.append(pd.read_csv(f"{dir_name}/fastlin_output.txt", sep='\t'))
    
fastlin_output = pd.concat(fastlin_output)

/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set5_240528
/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set6


In [161]:
df_check_lineages = df_PacBio_metadata[['Sample', '#sample', 'Directory', 'PacBio_FQ_PATH']].merge(fastlin_output, on='#sample')
len(df_check_lineages)

55

In [218]:
df_LR_SR.query("Illumina_ID=='MFS-160'").PacBio_FQ_PATH.values

array(['/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set3_230124/m64142_221216_110244.bc2091--bc2091.hifi_reads.fastq.gz'],
      dtype=object)

In [ ]:
a

In [162]:
df_check_lineages.Directory.unique()

array(['/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set5_240528',
       '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set6'],
      dtype=object)

In [163]:
df_check_lineages.query("Sample in @df_check_lineages_Set5_231017_samples.Sample")

,Sample,#sample,Directory,PacBio_FQ_PATH,data_type,k_cov,mixture,lineages,log_barcodes,log_errors


In [87]:
# df_check_lineages[['Sample', 'lineages']].merge(df_trust_patients[['SampleID', 'Coll2014', 'F2']], left_on='Sample', right_on='SampleID')

In [120]:
df_test = pd.read_excel("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports/2023-09-01_report_Farhat.v09.xlsx", sheet_name=None)

In [116]:
len(df_check_lineages_Set5_231017_samples)

28

In [131]:
Max_LR_metadata.query("SampleID in @df_check_lineages_Set5_231017_samples.Sample")

,Original_ID,PacBio_FQ_PATH,Illumina_PE_FQs_PATH,Dataset_Tag,SampleID


In [148]:
# df_test['seqorder_96'].loc[df_test['seqorder_96']['Strain_ID'].isin(df_check_lineages_Set5_231017_samples['Sample'])]

In [89]:
fastlin_output_Set5_231017_samples = pd.read_csv(f"/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set5_231017/fastlin_output.txt", sep='\t')

In [164]:
df_check_lineages_Set5_231017_samples = df_PacBio_metadata[['Sample', '#sample', 'Directory', 'PacBio_FQ_PATH']].merge(fastlin_output_Set5_231017_samples, on='#sample')
len(df_check_lineages_Set5_231017_samples)

28

In [165]:
# df_trust_patients[['SampleID', 'Coll2014', 'F2']].merge(df_check_lineages_Set5_231017_samples[['Sample', 'lineages']], left_on='SampleID', right_on='Sample')

In [316]:
df_addl_PacBio_matched = pd.concat([df_check_lineages, df_check_lineages_Set5_231017_samples]).rename(columns={'Sample': 'SampleID'}).dropna(how='all', axis=1)

df_addl_PacBio_matched = df_addl_PacBio_matched.merge(df_WGS[['Original_ID', 'SampleID']], how='left')

assert len(df_addl_PacBio_matched) == df_addl_PacBio_matched.SampleID.nunique()

df_addl_PacBio_matched['Dataset_Tag'] = [os.path.basename(dir_name) for dir_name in df_addl_PacBio_matched['Directory'].values]
df_addl_PacBio_matched['Illumina_ID'] = df_addl_PacBio_matched['SampleID']

df_addl_PacBio_matched.shape

(83, 12)

In [420]:
Max_LR_metadata = pd.read_csv("~/TRUST_data_processing/raw_data/230902.TRUST.LRandSR.Set1to4.106CI.InputWGS.PATHs.tsv", sep='\t')

Max_LR_metadata = Max_LR_metadata.rename(columns={'SampleID': 'Original_ID'}).merge(df_trust_patients[['Original_ID', 'SampleID']])

Max_LR_metadata.loc[Max_LR_metadata["PacBio_FQ_PATH"].str.contains('DownloadedData'), 'PacBio_FQ_PATH'] = Max_LR_metadata.loc[Max_LR_metadata["PacBio_FQ_PATH"].str.contains('DownloadedData')]['PacBio_FQ_PATH'].str.replace('/n/data1/hms/dbmi/farhat/mm774/DownloadedData/TRUST_PBSet1_PacBio_CCS', '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1')
Max_LR_metadata['Illumina_ID'] = Max_LR_metadata['Illumina_PE_FQs_PATH'].str.split(';').str[0].apply(lambda x: os.path.basename(x).split('_')[0])
del Max_LR_metadata['SampleID']

Max_LR_metadata = Max_LR_metadata.merge(df_PacBio_metadata[['Sample', 'PacBio_FQ_PATH']].rename(columns={'Sample': 'SampleID'}), on='PacBio_FQ_PATH')

df_new_combined = pd.concat([Max_LR_metadata[['Original_ID', 'PacBio_FQ_PATH', 'Dataset_Tag', 'SampleID', 'Illumina_ID']],
                             df_addl_PacBio_matched[['Original_ID', 'PacBio_FQ_PATH', 'Dataset_Tag', 'SampleID', 'Illumina_ID']]
                            ]).rename(columns={'SampleID': 'PacBio_ID'})

# add Illumina lineage info for comparison
df_new_combined = df_new_combined.merge(df_WGS[['SampleID', 'Coll2014', 'F2']].rename(columns={'SampleID': 'Illumina_ID'}), on='Illumina_ID', how='left').reset_index(drop=True)

# to merge with fastlin output
df_new_combined['#sample'] = [os.path.basename(fName).split('.fastq.gz')[0] for fName in df_new_combined['PacBio_FQ_PATH']]

df_new_combined['Illumina_PE_FQs_PATH'] = np.nan

for i, row in df_new_combined.iterrows():
    
    sample = row['Illumina_ID']
    
    fName_1 = f"/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumina/TRUST.SRWGS.FQs.AllSets.RenamedAndLanesMerged/{sample}/{sample}_R1.fastq.gz"
    fName_2 = f"/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumina/TRUST.SRWGS.FQs.AllSets.RenamedAndLanesMerged/{sample}/{sample}_R2.fastq.gz"
    
    if os.path.isfile(fName_1) and os.path.isfile(fName_2):    
        df_new_combined.loc[i, 'Illumina_PE_FQs_PATH'] = f"{fName_1};{fName_2}"
    else:
        print(f"No Illumina FQ files for {sample}")

df_new_combined.to_csv("~/TRUST_PacBio_Illumina.csv", index=False)        
len(df_new_combined), df_new_combined.Original_ID.nunique(), df_new_combined.PacBio_ID.nunique()

No Illumina FQ files for MFS-661
No Illumina FQ files for MFS-193


(189, 185, 189)

In [421]:
set(df_PacBio_metadata.PacBio_FQ_PATH) - set(df_new_combined.PacBio_FQ_PATH)

{'/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2003.ccs.fastq.gz',
 '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2004.ccs.fastq.gz',
 '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2005.ccs.fastq.gz',
 '/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2016.ccs.fastq.gz'}

In [438]:
df_new_combined.query("Original_ID not in @finished_asms")#.iloc[[0], :].to_csv( "/home/sak0914/MtbLongitudinalDiversity/lowAF_variant_calling/data/LR_SR_combined_data_to_run.tsv", sep='\t', index=False)

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Coll2014,F2,Illumina_PE_FQs_PATH
106,S0017-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-520,MFS-520,1.1.2,0.019246,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
107,S0063-02,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-550,MFS-550,2.2.1,0.010625,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
108,S0097-02,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-566,MFS-566,3,0.009978,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
109,S0120-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-583,MFS-583,2.2.1,0.010295,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
110,S0139-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-601,MFS-601,2.2.1,0.009904,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
...,...,...,...,...,...,...,...,...
184,S0349-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-429,MFS-429,4.1.2.1,0.010488,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
185,S0351-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-430,MFS-430,4.3.3,0.006222,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
186,S0351-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-431,MFS-431,4.3.3,0.007208,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
187,S0353-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-434,MFS-434,2.2.1,0.009987,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


108

In [424]:
len(df_new_combined), df_new_combined.Original_ID.nunique(), df_new_combined.dropna(subset='Original_ID').Original_ID.nunique()

(189, 185, 185)

In [432]:
del df_new_combined['#sample']
df_new_combined.to_csv( "/home/sak0914/MtbLongitudinalDiversity/lowAF_variant_calling/data/LR_SR_combined_data.tsv", sep='\t', index=False)

In [350]:
for fName in df_new_combined.PacBio_FQ_PATH.values:
    assert os.path.isfile(fName)

In [351]:
for sample in df_new_combined.Illumina_ID.values:
    if sample not in Illumina_samples:
        print(sample)

MFS-661
MFS-193


In [352]:
df_new_combined.loc[pd.isnull(df_new_combined['Original_ID'])]#.PacBio_FQ_PATH.values

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID
12,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-661,MFS-661
56,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-193,MFS-193
74,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-422,MFS-422
76,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-425,MFS-425


In [403]:
pacbio_fastlin_files = glob.glob(f"/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/*/fastlin_output.txt")
print(len(pacbio_fastlin_files))
fastlin_output = []

for fName in pacbio_fastlin_files:
    fastlin_output.append(pd.read_csv(fName, sep='\t'))
    
fastlin_output = pd.concat(fastlin_output)

6


In [404]:
df_lineage_check = df_new_combined.merge(fastlin_output[['#sample', 'k_cov', 'mixture', 'lineages']], on='#sample', how='left').reset_index(drop=True)

In [405]:
df_lineage_check['fastlin_lineage'] = np.nan

for i, row in df_lineage_check.iterrows():
    
    if not pd.isnull(row['lineages']):
        
        recombine_lineages = []
        
        split_lineages = row['lineages'].split(', ')
        
        for lineage in split_lineages:
            recombine_lineages.append(lineage.split(' (')[0])

        df_lineage_check.loc[i, 'fastlin_lineage'] = ','.join(np.sort(recombine_lineages))
        
    if not pd.isnull(row['Coll2014']):
        if ',' in row['Coll2014']:
            df_lineage_check.loc[i, 'Coll2014'] = ','.join(np.sort(row['Coll2014'].split(',')))

In [406]:
df_lineage_check.dropna(subset='fastlin_lineage').query("Coll2014 != fastlin_lineage")[['Illumina_ID', 'Coll2014', 'F2', 'PacBio_ID', 'fastlin_lineage']]

,Illumina_ID,Coll2014,F2,PacBio_ID,fastlin_lineage
31,MFS-43,4.1.2.1,0.021399,MFS-212,4.1.1.3
38,MFS-102,4.1.1.3,0.018669,MFS-232,"4.1.1.3,4.1.2.1"
40,MFS-107,4.1.1.1,0.007513,MFS-236,2.2.1
58,MFS-57,4.1.1.3,0.017603,MFS-224,"2.2.1.1,4.1.1.3"
59,MFS-103,4.1.1.1,0.007504,MFS-233,4.8
70,MFS-8,2.2.1.1,0.010502,MFS-220,2.2.1
85,MFS-150,4.4.1.1,0.005826,MFS-197,4.4.1.1.1
99,MFS-151,4.4.1.1,0.005072,MFS-242,4.4.1.1.1
127,MFS-84,2.2.1,0.177757,MFS-84,"2.2.1,4.1.1.3"
131,MFS-258,4.4.1.1,0.006545,MFS-258,4.4.1.1.1


In [ ]:
df_lineage_check

In [401]:
df_lineage_check.query("Illumina_ID=='MFS-107'")

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,SampleID,Coll2014,F2,#sample,k_cov,mixture,lineages,fastlin_lineage
40,S0189-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set2,MFS-236,MFS-107,MFS-107,4.1.1.1,0.007513,demultiplex.bc2067--bc2067.hifi_reads,93.0,no,2.2.1 (84),2.2.1


In [402]:
df_WGS.query("pid=='T0189'")

,pid,SampleID,Original_ID,Kraken_Unclassified_Percent,Mean_Depth,Median_Depth,Perc_Sites_10x,Perc_Sites_20x,F2,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,Study
589,T0189,MFS-107,S0189-01,0.52,260.304072,264.0,98.738601,98.601257,0.007513,4.1.1.1,4.1.i1.2.1,"xtype,westafrican1",NaN,4,4,TRUST


In [215]:
Illumina_samples = os.listdir("/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumina/TRUST.SRWGS.FQs.AllSets.RenamedAndLanesMerged")
len(Illumina_samples)

803

In [217]:
set(missing_samples).intersection(Illumina_samples)

{'MFS-422', 'MFS-425'}

In [220]:
df_WGS.query("SampleID in ['MFS-422', 'MFS-425']")

,pid,SampleID,Original_ID,Kraken_Unclassified_Percent,Mean_Depth,Median_Depth,Perc_Sites_10x,Perc_Sites_20x,F2,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,Study


In [222]:
df_WGS_full = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/Illumina_culture_WGS_summary.csv")

In [224]:
df_WGS_full.query("SampleID in ['MFS-422', 'MFS-425']")

,SampleID,Kraken_Unclassified_Percent,Mean_Depth,Median_Depth,Perc_Sites_10x,Perc_Sites_20x,F2,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,Freschi_Lineage_1,Freschi_Lineage_2,Culture_Mixed_Infection
289,MFS-422,77.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
292,MFS-425,85.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


# Hybrid Assembly Quality

In [10]:
df_LR_SR = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBio_Illumina_sample_mapping.csv")

hybrid_asms = glob.glob(f"{assembly_dir}/*/PB/Flye_Assembly/assembly.fasta")

len(hybrid_asms)

186

In [3]:
df_hybridASM_QC = pd.DataFrame(columns = ['Original_ID', 'numContigs', 'circContigLength', 'Flye_Coll2014', 'Flye_PP_Coll2014', 'NumPilonChanges'])

for i, fName in enumerate(hybrid_asms):
    
    fasta_file = [(seq.id, seq.seq) for seq in SeqIO.parse(fName, "fasta")]
    
    assembly_stats = pd.read_csv(f"{os.path.dirname(fName)}/assembly_info.txt", sep='\t')
    
    complete_circular_contigs = assembly_stats.loc[(assembly_stats['circ.']=='Y') & (assembly_stats['length'] >= 4000000)]
    
    if len(complete_circular_contigs) == 0:
        circContigLength = np.nan
    elif len(complete_circular_contigs) == 1:
        circContigLength = complete_circular_contigs['length'].values[0]
    else:
        raise ValueError(f"{fName} has more than 1 circular contig")
        
#     lengths_dist = [len(seq[1]) for seq in fasta_file]
    
#     if len(lengths_dist) > 1:
#         second_longest = np.sort(lengths_dist)[-2]
#     else:
#         second_longest = np.max(lengths_dist)
        
    try:
        match = re.search(r"S\d{4}-..", fName)
        sample = match.group()
    except:
        match = re.search(r"MFS-\d{2,3}", fName)
        sample = match.group()
    
    try:
        flye_lineage_call = pd.read_csv(f"{assembly_dir}/{sample}/LineageCalling/LineageCall_FlyeI3Asm/{sample}.AsmToRef.FlyeI3Asm.lineage_call.tsv", sep='\t').coll2014.str.replace('lineage', '').values[0]
        pilon_polished_lineage_call = pd.read_csv(f"{assembly_dir}/{sample}/LineageCalling/LineageCall_FlyeI3AsmPP/{sample}.AsmToRef.FlyeI3AsmPP.lineage_call.tsv", sep='\t').coll2014.str.replace('lineage', '').values[0]

        pilon_changes_file = f"{assembly_dir}/{sample}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/{sample}.Flye.I3Asm.PilonPolished.changes"

        with open(pilon_changes_file, "r") as file:
            num_changes = len(file.readlines())
    except:
        flye_lineage_call = ''
        pilon_polished_lineage_call = ''
        num_changes = np.nan
        
#     if num_SNP_changes > 0:
#         df_pilon_changes = pd.read_csv(pilon_changes_file, sep='\t', header=None)
#         df_pilon_changes = df_pilon_changes[0].str.split(' ', expand=True)
#         df_pilon_changes.columns = ['Orig_POS', 'New_POS', 'REF', 'ALT']
        
#         num_SNP_changes = len(df_pilon_changes.query("REF.str.len() == ALT.str.len()"))

    df_hybridASM_QC.loc[i, :] = [sample, 
                                 len(assembly_stats), 
                                 circContigLength,
                                 flye_lineage_call, 
                                 pilon_polished_lineage_call,
                                 num_changes
                                ]
    
extremely_fragmented_assemblies = df_hybridASM_QC.loc[pd.isnull(df_hybridASM_QC['NumPilonChanges'])].Original_ID.values

print(f"{len(extremely_fragmented_assemblies)} Extremely fragmented assemblies: {','.join(extremely_fragmented_assemblies)}")
    
df_hybridASM_QC = df_hybridASM_QC.merge(df_LR_SR[['Original_ID', 'PacBio_ID', 'Illumina_ID', 'Illumina_F2', 'Illumina_Coll2014']].drop_duplicates())
print(len(df_hybridASM_QC), df_hybridASM_QC.Original_ID.nunique())

0 Extremely fragmented assemblies: 
190 186


In [81]:
fastlin_files = glob.glob("/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/*/fastlin_output.txt")

fastlin_output = []

for fName in fastlin_files:
    fastlin_output.append(pd.read_csv(fName, sep='\t'))
    
fastlin_output = pd.concat(fastlin_output)
fastlin_output['lineages'] = fastlin_output['lineages'].fillna(fastlin_output['log_barcodes'])

In [83]:
df = pd.read_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data.tsv", sep='\t')

to_fix_lst = df.loc[pd.isnull(df['Original_ID'])].PacBio_ID.values

df_og = df.query("PacBio_ID not in @to_fix_lst")
df_fix = df.query("PacBio_ID in @to_fix_lst").dropna(axis=1, how='all')

df_fix.loc[df_fix['PacBio_ID']=='MFS-193', 'Original_ID'] = 'S0161-01'
df_fix.loc[df_fix['PacBio_ID']=='MFS-422', 'Original_ID'] = 'S0344-01'
df_fix.loc[df_fix['PacBio_ID']=='MFS-425', 'Original_ID'] = 'S0346-01'
df_fix.loc[df_fix['PacBio_ID']=='MFS-661', 'Original_ID'] = 'S0360-01'

del_cols = ['Illumina_ID', 'F2', 'Coll2014', 'Kraken_Unclassified_Percent', 'PacBio_fastlin_mixture', 'PacBio_fastlin_output', 'PacBio_fastlin']

for col in del_cols:
    if col in df_fix.columns:
        del df_fix[col]

In [84]:
df_Illumina_to_merge = df_trust_patients[['pid', 'SampleID', 'Original_ID']].merge(df_Illumina_WGS, on='SampleID')

df_fix = df_fix.merge(df_Illumina_to_merge[['Original_ID', 'SampleID', 'F2', 'Coll2014', 'Kraken_Unclassified_Percent']].rename(columns={'SampleID': 'Illumina_ID'}), on='Original_ID').merge(fastlin_output[['#sample', 'mixture', 'lineages']].rename(columns={'mixture': 'PacBio_fastlin_mixture', 'lineages': 'PacBio_fastlin_output'}), on='#sample')

In [97]:
for i, row in df_fix.iterrows():
    
    split_lineages = row['PacBio_fastlin_output'].split(', ')
    
    combined_lineages = [lineage.split(' (')[0] for lineage in split_lineages]
    
    df_fix.loc[i, 'PacBio_fastlin'] = ','.join(combined_lineages)

In [105]:
df_new = pd.concat([df_og, df_fix[df_og.columns]])

In [24]:
df.query("Original_ID=='S0185-01'").PacBio_FQ_PATH.values

array(['/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set3_230124/m64142_221216_110244.bc2027--bc2027.hifi_reads.fastq.gz'],
      dtype=object)

In [22]:
Max_LR_metadata.query("SampleID=='S0185-01'")

,SampleID,PacBio_FQ_PATH,Illumina_PE_FQs_PATH,Dataset_Tag
59,S0185-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,TRUST_PB_Set3


In [23]:
df_hybridASM_QC.query("numContigs==1 & circContigLength >= 4000000 & Flye_Coll2014 == Flye_PP_Coll2014")

,Original_ID,numContigs,circContigLength,secondLongestContig,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,F2,Coll2014
0,S0001-01,1,4416220,4416220,4.3.2.1,4.3.2.1,0,MFS-331,MFS-331,0.010162,4.3.2.1
1,S0004-01,1,4416486,4416486,2.2.1.1,2.2.1.1,2,MFS-218,MFS-1,0.011212,2.2.1.1
2,S0004-08,1,4416487,4416487,2.2.1.1,2.2.1.1,1,MFS-219,MFS-2,0.010401,2.2.1.1
3,S0007-01,1,4423153,4423153,2.2.1,2.2.1,2,MFS-3,MFS-3,0.008362,2.2.1
4,S0008-01,1,4404929,4404929,4.1.1.1,4.1.1.1,7,MFS-195,MFS-4,0.007025,4.1.1.1
...,...,...,...,...,...,...,...,...,...,...,...
184,S0403-01,1,4408331,4408331,4.4.1.1,4.4.1.1,0,MFS-677,MFS-677,0.003829,4.4.1.1
185,S0419-01,1,4364836,4364836,4.3.4.2.1,4.3.4.2.1,0,MFS-690,MFS-690,0.003732,4.3.4.2.1
186,S0427-02,1,4418492,4418492,2.2.1,2.2.1,2,MFS-696,MFS-696,0.108637,2.2.1
187,S0430-02,1,4415828,4415828,2.2.1,2.2.1,0,MFS-812,MFS-812,0.008941,2.2.1


In [158]:
df_hybridASM_QC.query("numContigs==1 & circContigLength >= 4000000 & Flye_Coll2014 == Flye_PP_Coll2014 & F2 <= @F2_max").sort_values('NumPilonChanges')

,Original_ID,numContigs,circContigLength,secondLongestContig,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,F2,Coll2014
0,S0001-01,1,4416220,4416220,4.3.2.1,4.3.2.1,0,MFS-331,MFS-331,0.010162,4.3.2.1
138,S0292-01,1,4401996,4401996,4.8,4.8,0,MFS-310,MFS-310,0.010262,4.8
137,S0291-09,1,4426315,4426315,4.1.2,4.1.2,0,MFS-309,MFS-309,0.021034,4.1.2
136,S0291-01,1,4426315,4426315,4.1.2,4.1.2,0,MFS-308,MFS-308,0.018474,4.1.2
135,S0290-08,1,4416520,4416520,2.2.1.1,2.2.1.1,0,MFS-266,MFS-266,0.012428,2.2.1.1
...,...,...,...,...,...,...,...,...,...,...,...
11,S0031-01,1,4424460,4424460,2.2.1,2.2.1,8,MFS-217,MFS-16,0.010009,2.2.1
80,S0188-02,1,4403238,4403238,4.3.3,4.3.3,37,MFS-192,MFS-106,0.005929,4.3.3
106,S0227-07,1,4404783,4404783,4.1.1.1,4.1.1.1,48,MFS-136,MFS-136,0.007964,4.1.1.1
105,S0227-07,1,4404783,4404783,4.1.1.1,4.1.1.1,48,MFS-241,MFS-136,0.007964,4.1.1.1


In [117]:
Max_ASM_QC[['SampleID', 'NumContigs', 'numContigs_Complete', 'circContig_Length',
       'circContig_Cov', 'Flye_EstimatedCov', 'Flye_ReadLen_N50',
       'Flye_ReadLen_N90', 'Lineage_Asm', 'Lineage_AsmPP', 'NumChanges_PilonPolished']].query("NumContigs == 1 & circContig_Length >= 4000000 & Lineage_Asm == Lineage_AsmPP").sort_values('NumChanges_PilonPolished')

,SampleID,NumContigs,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,NumChanges_PilonPolished
50,S0276-01,1,1,4424401,110,111,8003,4697,lineage2.2.1,lineage2.2.1,0
38,S0180-01,1,1,4416436,82,81,7651,4588,lineage2.2.1.1,lineage2.2.1.1,0
96,S0049-08,1,1,4405000,132,133,11231,6437,lineage4.1.1.1,lineage4.1.1.1,0
41,S0284-01,1,1,4412753,95,95,8339,4807,lineage2.2.1.1,lineage2.2.1.1,0
83,S0285-08,1,1,4404269,94,95,8444,4895,lineage4.3.2.1,lineage4.3.2.1,0
...,...,...,...,...,...,...,...,...,...,...,...
77,S0008-01,1,1,4404929,107,108,10320,6068,lineage4.1.1.1,lineage4.1.1.1,7
7,S0031-01,1,1,4424460,142,141,10107,6609,lineage2.2.1,lineage2.2.1,8
105,S0188-02,1,1,4403238,43,42,9253,5701,lineage4.3.3,lineage4.3.3,37
91,S0227-07,1,1,4404783,121,121,9825,6197,lineage4.1.1.1,lineage4.1.1.1,48


In [106]:
df_new

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Kraken_Unclassified_Percent,F2,Coll2014,#sample,PacBio_fastlin_mixture,PacBio_fastlin_output,PacBio_fastlin,Illumina_FQ_PATH
0,S0007-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-3,MFS-3,0.41,0.008362,2.2.1,TRUST.bc2001.ccs,no,2.2.1 (51),2.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
1,S0256-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-172,MFS-172,0.83,0.012057,2.2.1.1,TRUST.bc2018.ccs,no,2.2.1.1 (25),2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
2,S0252-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-168,MFS-168,1.35,0.477297,"2.2.1.1,4.1.1.3",TRUST.bc2017.ccs,yes,"4.1.1.3 (21), 2.2.1.1 (17)","2.2.1.1,4.1.1.3",/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
3,S0130-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-62,MFS-62,0.52,0.009158,2.2.1.1,TRUST.bc2014.ccs,no,2.2.1.1 (35),2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
4,S0107-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-59,MFS-59,0.56,0.011810,2.2.1.1,TRUST.bc2011.ccs,no,2.2.1.1 (39),2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,S0227-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-136,MFS-136,0.69,0.007964,4.1.1.1,TRUST.bc2016.ccs,no,4.1.1.1 (8),4.1.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
0,S0360-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_240528,MFS-661,MFS-770,0.42,0.011217,2.2.1.1,m64142_240409_061126.bc2013--bc2013.hifi_reads,no,"1.1.2 (20), 1.2.1.2.1 (14)","1.1.2,1.2.1.2.1",NaN
1,S0161-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-193,MFS-80,0.56,0.010541,2.2.1.1,m64142_230421_105523.bc2074--bc2074.hifi_reads,no,2.2.1.1 (37),2.2.1.1,NaN
2,S0344-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-422,MFS-422,77.35,NaN,NaN,m64142_230928_115158.bc2056--bc2056.hifi_reads,no,2.2.1.1 (10),2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


In [112]:
df_keep = df_new.rename(columns={'Kraken_Unclassified_Percent': 'Illumina_Kraken_Unclassified_Percent',
                         'F2': 'Illumina_F2',
                         'Coll2014': 'Illumina_Coll2014',
                        })

del df_keep['PacBio_fastlin_mixture']
del df_keep['PacBio_fastlin_output']
# del df_keep['PacBio_fastlin']
del df_keep['#sample']

# df_keep = df_keep.merge(df_hybridASM_QC.iloc[:, :-2], on=['Original_ID', 'PacBio_ID', 'Illumina_ID'], how='left')
# del df_keep['NumPilonChanges']

df_keep[['Original_ID', 'PacBio_ID', 'Dataset_Tag', 'PacBio_FQ_PATH', 'PacBio_fastlin', 'Illumina_ID', 'Illumina_Kraken_Unclassified_Percent', 'Illumina_F2', 'Illumina_Coll2014']].sort_values('Original_ID').to_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBio_Illumina_sample_mapping.csv", index=False)

In [121]:
df_whoo = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBio_Illumina_sample_mapping.csv")
len(df_whoo), sum(pd.isnull(df_whoo['Original_ID'])), df_whoo.Original_ID.nunique()

(193, 0, 189)

In [122]:
df_whoo.query("Original_ID=='S0077-01'")

,Original_ID,PacBio_ID,Dataset_Tag,PacBio_FQ_PATH,PacBio_fastlin,Illumina_ID,Illumina_Kraken_Unclassified_Percent,Illumina_F2,Illumina_Coll2014
32,S0077-01,MFS-43,TRUST_PB_Set1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.2.1,MFS-43,0.47,0.021399,4.1.2.1
33,S0077-01,MFS-212,TRUST_PB_Set2,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.1.3,MFS-43,0.47,0.021399,4.1.2.1


In [126]:
set(df_whoo.Original_ID) - set(finished_ASMs)

{'S0161-01', 'S0344-01', 'S0346-01', 'S0360-01'}

In [127]:
set(df_whoo.query("PacBio_fastlin==Illumina_Coll2014").Original_ID) - set(finished_ASMs)

{'S0161-01'}

In [130]:
df_whoo.query("Original_ID=='S0161-01'").to_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data_to_run.tsv", sep='\t', index=False)

In [128]:
df_whoo.query("Original_ID in ['S0344-01', 'S0346-01', 'S0360-01']")

,Original_ID,PacBio_ID,Dataset_Tag,PacBio_FQ_PATH,PacBio_fastlin,Illumina_ID,Illumina_Kraken_Unclassified_Percent,Illumina_F2,Illumina_Coll2014
171,S0344-01,MFS-422,TRUST_PacBio_Set5_231017,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,2.2.1.1,MFS-422,77.35,NaN,NaN
173,S0346-01,MFS-425,TRUST_PacBio_Set5_231017,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,"2.2.1,4.9,8",MFS-425,85.23,NaN,NaN
180,S0360-01,MFS-661,TRUST_PacBio_Set5_240528,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,"1.1.2,1.2.1.2.1",MFS-770,0.42,0.011217,2.2.1.1


In [133]:
df_trust_patients.query("pid in ['T0344'] & Original_ID != 'S0344-01'")[['pid', 'Original_ID', 'SampleID', 'Coll2014', 'F2']]

,pid,Original_ID,SampleID,Coll2014,F2
145,T0344,S0344-02,MFS-766,2.2.1.1,0.011588


In [135]:
df_trust_patients.query("pid in ['T0346'] & Original_ID != 'S0346-01'")[['pid', 'Original_ID', 'SampleID', 'Coll2014', 'F2']]

,pid,Original_ID,SampleID,Coll2014,F2
150,T0346,S0346-02,MFS-768,2.2.1.1,0.011578
151,T0346,S0346-09,MFS-769,2.2.1,0.012190
152,T0346,S0346-08,MFS-426,2.2.1.1,0.011563


In [164]:
df_hybridASM_QC.query("Original_ID=='S0077-01'")

,Original_ID,numContigs,circContigLength,secondLongestContig,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,F2,Coll2014
32,S0077-01,2,3282798,1138742,4.1.2.1,4.1.2.1,1,MFS-212,MFS-43,0.021399,4.1.2.1
33,S0077-01,2,3282798,1138742,4.1.2.1,4.1.2.1,1,MFS-43,MFS-43,0.021399,4.1.2.1


In [171]:
df_LR_SR.query("Original_ID=='S0077-01' & PacBio_ID=='MFS-43'").PacBio_FQ_PATH.values

array(['/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio/TRUST_PacBio_Set1/TRUST.bc2003.ccs.fastq.gz'],
      dtype=object)

In [173]:
df_LR_SR.query("Kraken_Unclassified_Percent > 20")

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Kraken_Unclassified_Percent,F2,Coll2014,#sample,PacBio_fastlin_mixture,PacBio_fastlin_output,PacBio_fastlin,Illumina_FQ_PATH
180,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-422,MFS-422,77.35,NaN,NaN,m64142_230928_115158.bc2056--bc2056.hifi_reads,no,2.2.1.1 (10),2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...
182,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-425,MFS-425,85.23,NaN,NaN,m64142_230928_115158.bc2058--bc2058.hifi_reads,no,NaN,NaN,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...


In [177]:
Four samples had 

185

In [166]:
df_keep.query("Original_ID=='S0077-01'")

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Illumina_Kraken_Unclassified_Percent,Illumina_F2,Illumina_Coll2014,Illumina_FQ_PATH,numContigs,circContigLength,secondLongestContig,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges
31,S0077-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set2,MFS-212,MFS-43,0.47,0.021399,4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,2,3282798,1138742,4.1.2.1,4.1.2.1,1
189,S0077-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-43,MFS-43,0.47,0.021399,4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,2,3282798,1138742,4.1.2.1,4.1.2.1,1


In [163]:
df_keep

,Original_ID,PacBio_FQ_PATH,Dataset_Tag,PacBio_ID,Illumina_ID,Illumina_Kraken_Unclassified_Percent,Illumina_F2,Illumina_Coll2014,Illumina_FQ_PATH,numContigs,circContigLength,secondLongestContig,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges
0,S0007-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-3,MFS-3,0.41,0.008362,2.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,1,4423153,4423153,2.2.1,2.2.1,2
1,S0256-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-172,MFS-172,0.83,0.012057,2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,3,2215673,1657915,2.2.1.1,2.2.1.1,4
2,S0252-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-168,MFS-168,1.35,0.477297,"2.2.1.1,4.1.1.3",/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,14,4296871,94629,2.2.1.1,2.2.1.1,52
3,S0130-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-62,MFS-62,0.52,0.009158,2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,4,2225661,1558961,2.2.1.1,2.2.1.1,0
4,S0107-07,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-59,MFS-59,0.56,0.011810,2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,1,4416464,4416464,2.2.1.1,2.2.1.1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,S0358-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PacBio_Set5_231017,MFS-443,MFS-443,0.41,0.010711,2.2.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,1,4413511,4413511,2.2.1.1,2.2.1.1,0
189,S0077-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-43,MFS-43,0.47,0.021399,4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,2,3282798,1138742,4.1.2.1,4.1.2.1,1
190,S0077-06,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-44,MFS-44,0.66,0.021498,4.1.2.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,1,4421214,4421214,4.1.2.1,4.1.2.1,0
191,S0082-01,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,TRUST_PB_Set1,MFS-47,MFS-47,0.62,0.012186,3.1.1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_Illumi...,4,4424006,50914,3.1.1,3.1.1,30


# Match Assemblies with Illumina Samples

### If an Illumina sample doesn't have an assembly, but another timepoint of the same person does, use that 

### Check again that the lineages match and only include samples with Illumina F2 ≤ 0.03 

In [57]:
df_hybridASM_QC.query("numContigs==1 & circContigLength >= 4000000 & Flye_Coll2014 == Flye_PP_Coll2014 & Illumina_F2 > @F2_max")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014
88,S0198-02,1,4416530,2.2.1.1,2.2.1.1,0,MFS-377,MFS-377,0.108434,2.2.1.1
187,S0427-02,1,4418492,2.2.1,2.2.1,2,MFS-696,MFS-696,0.108637,2.2.1


In [58]:
df_hybridASM_QC.query("numContigs==1 & circContigLength >= 4000000 & Flye_Coll2014 != Flye_PP_Coll2014")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014
78,S0185-01,1,4401996,4.8,4.1.1.1,1044,MFS-233,MFS-103,0.007504,4.1.1.1


In [60]:
df_LR_SR.query("Original_ID=='S0185-01'")

,Original_ID,PacBio_ID,Dataset_Tag,PacBio_FQ_PATH,PacBio_fastlin,Illumina_ID,Illumina_Kraken_Unclassified_Percent,Illumina_F2,Illumina_Coll2014
78,S0185-01,MFS-233,TRUST_PB_Set3,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.8,MFS-103,0.81,0.007504,4.1.1.1


In [4]:
df_hybridASM_QC['pid'] = df_hybridASM_QC['Original_ID'].str.split('-').str[0].str.replace('S', 'T')

df_hybridASM_highQuality = df_hybridASM_QC.query("numContigs==1 & circContigLength >= 4000000 & Flye_Coll2014 == Flye_PP_Coll2014").query("PacBio_ID not in ['MFS-212', 'MFS-44', 'MFS-214', 'MFS-136']").reset_index(drop=True)
print(len(df_hybridASM_highQuality))

df_hybridASM_highQuality = df_hybridASM_highQuality.merge(df_trust_patients.dropna(subset='Lineage')[['Original_ID', 'pid']].drop_duplicates(), how='left').reset_index(drop=True)

print(f"{len(df_hybridASM_highQuality)} high-quality hybrid assemblies across {df_hybridASM_highQuality.pid.nunique()} pids")

154
154 high-quality hybrid assemblies across 117 pids


In [88]:
df_hybridASM_QC.loc[pd.isnull(df_hybridASM_QC['circContigLength'])].query("numContigs==1")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014
74,S0171-01,1,NaN,4.3.2.1,4.3.2.1,0,MFS-90,MFS-90,0.008858,4.3.2.1
124,S0272-02,1,NaN,2.2.1,2.2.1,0,MFS-383,MFS-383,0.009333,2.2.1


In [95]:
len(df_hybridASM_QC), df_hybridASM_QC.Original_ID.nunique()

(190, 186)

In [113]:
df_hybridASM_QC.query("PacBio_ID not in ['MFS-212', 'MFS-44', 'MFS-214', 'MFS-136']").set_index('pid').reset_index().sort_values(['pid', 'Original_ID']).to_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/hybridASM_QC.csv", index=False)

In [114]:
df_hybridASM_highQuality.circContigLength.min(), df_hybridASM_highQuality.circContigLength.max()

(4364836, 4439444)

In [119]:
df_hybridASM_QC.query("PacBio_ID not in ['MFS-212', 'MFS-44', 'MFS-214', 'MFS-136']").query("Flye_Coll2014 != Flye_PP_Coll2014")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014,pid
9,S0018-06,2,NaN,2.2.1,2.2.1.1,286,MFS-220,MFS-8,0.010502,2.2.1.1,T0018
50,S0106-08,2,4416632,2.2.1.1,4.1.1.3,1568,MFS-224,MFS-57,0.017603,4.1.1.3,T0106
78,S0185-01,1,4401996,4.8,4.1.1.1,1044,MFS-233,MFS-103,0.007504,4.1.1.1,T0185
82,S0189-01,2,NaN,2.2.1,4.1.1.1,1620,MFS-236,MFS-107,0.007513,4.1.1.1,T0189


In [148]:
df_hybridASM_QC.query("Original_ID=='S0082-01'")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014,pid
36,S0082-01,2,NaN,3.1.1,3.1.1,1,MFS-214,MFS-47,0.012186,3.1.1,T0082
37,S0082-01,2,NaN,3.1.1,3.1.1,1,MFS-47,MFS-47,0.012186,3.1.1,T0082


In [83]:
Illumina_samples = os.listdir("/n/data1/hms/dbmi/farhat/Sanjana/TRUST_lowAF")
len(Illumina_samples)

772

In [57]:
df_personal_ASM_for_aln = pd.DataFrame(columns = ['pid', 'Original_ID', 'SampleID', 'ASM_original', 'SameSample'])
samples_no_personal_ref = []

for i, row in df_trust_patients.dropna(subset='Lineage').iterrows():
    
    sample = row['SampleID']
    pid = row['pid']
    original_ID = row['Original_ID']
    
    if original_ID in df_hybridASM_highQuality.Original_ID.values:
        
        asm_fName = f"{assembly_dir}/{original_ID}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/{original_ID}.Flye.I3Asm.PilonPolished.fasta"
        assert os.path.isfile(asm_fName)
        
        df_personal_ASM_for_aln.loc[i, :] = [pid, original_ID, sample, asm_fName, 1]
        
    else:
        
        if pid in df_hybridASM_highQuality.pid.values:
            
            # take the first one. There shouldn't be multiple others (that would be 3 PacBio samples person)
            another_ID = df_hybridASM_highQuality.query("pid==@pid").sort_values('Original_ID').drop_duplicates('Original_ID')
            
            if len(another_ID) > 1:
                print(f"{sample} has more than 2 PacBio timepoints")
            
            another_ID = another_ID.Original_ID.values[0]
            
            asm_fName = f"{assembly_dir}/{another_ID}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/{another_ID}.Flye.I3Asm.PilonPolished.fasta"
            assert os.path.isfile(asm_fName)
            
            # then need to check that the lineages match, but only for unmixed samples. Sometimes they do not due to sample mislabeling
            # if df_trust_patients.query("SampleID==@sample").F2.values[0] <= F2_max:
                
            SR_lineage = df_trust_patients.query("SampleID==@sample")['Coll2014'].values[0]
            LR_lineage = df_hybridASM_highQuality.query("Original_ID==@another_ID")['Flye_PP_Coll2014'].values[0]

            # there shouldn't be any lineage mixing among LR assemblies because if it was severe, then there would have been multiple contigs in the assembly, and we filtered those out above
            if ',' in SR_lineage:
                SR_lineages = SR_lineage.split(',')                
                
                if len(set(SR_lineages).intersection([LR_lineage])) >= 1:
                    print(f"Illumina sample {sample} has lineage mixing but a matched PacBio sample")
                    df_personal_ASM_for_aln.loc[i, :] = [pid, another_ID, sample, asm_fName, 0]
            else:
                if SR_lineage == LR_lineage:
                    df_personal_ASM_for_aln.loc[i, :] = [pid, another_ID, sample, asm_fName, 0]
                else:
                    print(f"Illumina sample {sample} does not have a matched PacBio sample with the same lineage")

        else:
            samples_no_personal_ref.append(sample)
      
print(f"\n{df_personal_ASM_for_aln['ASM_original'].nunique()} total personal reference genomes")
print(f"{df_personal_ASM_for_aln.SampleID.nunique()} Illumina samples across {df_personal_ASM_for_aln.pid.nunique()} pids have a personal reference")
print(f"{len(samples_no_personal_ref)} Illumina samples do not have a personal reference")

# add in the Illumina ID that matches the hybrid asembly (meaning the Illumina sample that was used to polish the assembly)
df_personal_ASM_for_aln = df_personal_ASM_for_aln.merge(df_hybridASM_highQuality[['Original_ID', 'Illumina_ID']])

Illumina sample MFS-769 does not have a matched PacBio sample with the same lineage
Illumina sample MFS-781 does not have a matched PacBio sample with the same lineage
Illumina sample MFS-809 does not have a matched PacBio sample with the same lineage
Illumina sample MFS-631 has lineage mixing but a matched PacBio sample
Illumina sample MFS-350 has lineage mixing but a matched PacBio sample
Illumina sample MFS-317 has lineage mixing but a matched PacBio sample
Illumina sample MFS-313 has lineage mixing but a matched PacBio sample

154 total personal reference genomes
179 Illumina samples across 117 pids have a personal reference
590 Illumina samples do not have a personal reference


In [117]:
df_personal_ASM_for_aln.query("SampleID=='MFS-630'")

,pid,Original_ID,SampleID,ASM_original,SameSample,Illumina_ID,ASM
31,T0204,S0204-m5,MFS-630,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-632,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...


In [122]:
df_personal_ASM_for_aln.query("SampleID=='MFS-631'")

,pid,Original_ID,SampleID,ASM_original,SameSample,Illumina_ID,ASM
32,T0204,S0204-m5,MFS-631,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-632,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...


In [77]:
df_personal_ASM_for_aln.query("SampleID=='MFS-769'")

,pid,Original_ID,SampleID,ASM_original,SameSample,Illumina_ID,ASM


In [118]:
df_check_lineage_mixing = df_personal_ASM_for_aln.merge(df_trust_patients[['SampleID', 'F2', 'Coll2014']].dropna())

In [123]:
len(df_check_lineage_mixing.query("F2 <= @F2_max"))

172

In [119]:
set(df_check_lineage_mixing.query("F2 <= @F2_max").SampleID) - set(df_trust_patients.query("F2 <= @F2_max & SampleID in @ground_truth_samples").SampleID)

set()

In [120]:
df_check_lineage_mixing.query("SampleID in ['MFS-630', 'MFS-631']")

,pid,Original_ID,SampleID,ASM_original,SameSample,Illumina_ID,ASM,F2,Coll2014
31,T0204,S0204-m5,MFS-630,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-632,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...,0.147559,2.2.1
32,T0204,S0204-m5,MFS-631,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-632,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...,0.314259,"2.2.1,3.1.1"


In [107]:
df_trust_patients.query("SampleID in ['MFS-630', 'MFS-631']")

,pid,Original_ID,SampleID,SampleLib,Pacbio,Comment,quality,Cov Any Mean,Cov Unam Perc,Perc. Reads Mapped,...,duration_of_use_meth,duration_of_use_mandrax,lca,fib4,prevtb_outcome,F2,Coll2014,Freschi2020,Lineage,Sampling_Week
395,T0204,S0204-01,MFS-630,MFS-630_lib66023,NaN,NaN,1.0,357.99,1.0,0.9852,...,NaN,NaN,2.0,0.238897,NaN,0.147559,2.2.1,2.2.1.1.1,2,1
396,T0204,S0204-08,MFS-631,MFS-631_lib66024,NaN,NaN,1.0,274.85,1.0,0.9835,...,NaN,NaN,2.0,0.238897,NaN,0.314259,"2.2.1,3.1.1","2.2.1.1.1,3.1.1.i1","2,3",8


In [95]:
ground_truth_samples = os.listdir(personal_ref_dir)
clonal = True

num_ground_truth_samples = len(df_trust_patients.query("SampleID in @ground_truth_samples"))

if clonal:
    num_ground_truth_samples = len(df_trust_patients.query("F2 <= @F2_max & SampleID in @ground_truth_samples"))

non_ground_truth_samples = set(os.listdir(H37Rv_ref_dir)) - set(ground_truth_samples)
num_non_ground_truth_samples = len(df_trust_patients.query("F2 <= @F2_max & SampleID in @non_ground_truth_samples"))

print(f"{num_ground_truth_samples} ground truth samples, {num_non_ground_truth_samples} validation samples")

172 ground truth samples, 570 validation samples


In [87]:
df_check_lineage_mixing.query("F2 > 0.03")

,pid,Original_ID,SampleID,ASM_original,SameSample,Illumina_ID,ASM,F2,Coll2014
20,T0427,S0427-02,MFS-696,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,1,MFS-696,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...,0.108637,2.2.1
76,T0198,S0198-02,MFS-377,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,1,MFS-377,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...,0.108434,2.2.1.1
157,T0311,S0311-08,MFS-317,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-318,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...,0.012310,2.2.1.1


In [80]:
df_trust_patients.query("SampleID in ['MFS-769', 'MFS-781', 'MFS-809']")[['pid', 'SampleID', 'F2', 'Coll2014']]

,pid,SampleID,F2,Coll2014
151,T0346,MFS-769,0.012190,2.2.1
179,T0398,MFS-781,0.005280,4.8
217,T0427,MFS-809,0.010843,2.2.1.1


In [48]:
df_personal_ASM_for_aln.SameSample.value_counts()

SameSample
1    155
0     24
Name: count, dtype: int64

In [166]:
df_what = df_personal_ASM_for_aln.query("SameSample==1").reset_index(drop=True)

df_what.iloc[df_what.index.values[df_what.duplicated(subset='ASM', keep=False)]]

,pid,SampleID,ASM,SameSample
1,T0206,MFS-878,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,1
2,T0206,MFS-635,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,1


In [178]:
df_trust_patients.query("pid in ['T0192', 'T0204']")[['pid', 'Original_ID', 'SampleID', 'F2', 'Coll2014']]

,pid,Original_ID,SampleID,F2,Coll2014
39,T0192,S0192-05,MFS-877,0.007714,4.1.1.1
40,T0192,S0192-05,MFS-755,0.008691,4.1.1.1
41,T0192,S0192-02,MFS-194,0.010089,4.1.1.1
395,T0204,S0204-01,MFS-630,0.147559,2.2.1
396,T0204,S0204-08,MFS-631,0.314259,"2.2.1,3.1.1"
397,T0204,S0204-m5,MFS-632,0.008530,2.2.1


In [176]:
df_personal_ASM_for_aln.query("SameSample==0")

,pid,SampleID,ASM,SameSample
0,T0192,MFS-877,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
1,T0192,MFS-755,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
13,T0283,MFS-761,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
15,T0345,MFS-767,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
17,T0346,MFS-768,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
22,T0008,MFS-513,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
29,T0166,MFS-616,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
31,T0204,MFS-630,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
32,T0204,MFS-631,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0
36,T0211,MFS-640,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0


In [168]:
df_trust_patients.query("SampleID in ['MFS-878', 'MFS-635']")[['pid', 'Original_ID', 'SampleID', 'F2', 'Coll2014']]

,pid,Original_ID,SampleID,F2,Coll2014
42,T0206,S0206-01,MFS-878,0.010581,4.1.1.3
43,T0206,S0206-01,MFS-635,0.010560,4.1.1.3


In [169]:
df_trust_patients.query("pid=='T0206'")[['pid', 'Original_ID', 'SampleID', 'F2', 'Coll2014']]

,pid,Original_ID,SampleID,F2,Coll2014
42,T0206,S0206-01,MFS-878,0.010581,4.1.1.3
43,T0206,S0206-01,MFS-635,0.010560,4.1.1.3


In [172]:
df_hybridASM_highQuality.query("Original_ID=='S0206-01'")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014,pid
69,S0206-01,1,4402211,4.1.1.3,4.1.1.3,0,MFS-635,MFS-635,0.01056,4.1.1.3,T0206


In [158]:
df_trust_patients.dropna(subset='Lineage').pid.nunique()

424

In [58]:
for i, row in df_personal_ASM_for_aln.iterrows():
    
    sample = row['SampleID']
    ASM_sample = row['Illumina_ID']

    if row['SameSample'] == 1 and sample == row['Illumina_ID']:
        os.makedirs(f"{personal_ref_dir}/{sample}/assembly", exist_ok=True)

        if not os.path.isfile(f"{personal_ref_dir}/{sample}/assembly/{sample}.fasta"):
            shutil.copy(row['ASM_original'], f"{personal_ref_dir}/{sample}/assembly/{sample}.fasta")
            
    assert os.path.isfile(f"{personal_ref_dir}/{ASM_sample}/assembly/{ASM_sample}.fasta")
    df_personal_ASM_for_aln.loc[i, 'ASM'] = f"{personal_ref_dir}/{ASM_sample}/assembly/{ASM_sample}.fasta"
            
df_personal_ASM_for_aln = df_personal_ASM_for_aln.reset_index(drop=True)
print(len(df_personal_ASM_for_aln))

df_personal_ASM_for_aln[['SampleID', 'ASM_original', 'ASM']].to_csv("../lowAF_variant_calling/data/TRUST_SR_samples_personal_genomes.tsv", sep='\t', header=None, index=False)

179


In [73]:
df_personal_ASM_for_aln[['SampleID', 'ASM_original', 'ASM']].to_csv("../lowAF_variant_calling/data/TRUST_SR_samples_personal_genomes.tsv", sep='\t', header=None, index=False)

,SampleID,ASM_original,ASM
23,MFS-4,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...
92,MFS-26,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...
132,MFS-160,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...
154,MFS-321,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...
155,MFS-322,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_per...


In [37]:
df_personal_ASM_for_aln.query("SameSample==1 & SampleID != Illumina_ID")

,pid,Original_ID,SampleID,ASM,SameSample,Illumina_ID
3,T0206,S0206-01,MFS-878,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,1,MFS-635


In [43]:
for i, row in df_personal_ASM_for_aln.iterrows():
    assert os.path.isfile(f"{personal_ref_dir}/{row['Illumina_ID']}/assembly/{row['Illumina_ID']}.fasta")

In [40]:
df_personal_ASM_for_aln.query("SameSample==0")

,pid,Original_ID,SampleID,ASM,SameSample,Illumina_ID
0,T0192,S0192-02,MFS-877,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-194
1,T0192,S0192-02,MFS-755,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-194
13,T0283,S0283-01,MFS-761,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-324
15,T0345,S0345-08,MFS-767,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-424
17,T0346,S0346-08,MFS-768,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-426
22,T0008,S0008-01,MFS-513,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-4
29,T0166,S0166-07,MFS-616,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-199
31,T0204,S0204-m5,MFS-630,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-632
32,T0204,S0204-m5,MFS-631,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-632
36,T0211,S0211-08,MFS-640,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,0,MFS-121


In [39]:
df_personal_ASM_for_aln.query("SampleID=='MFS-635'")

,pid,Original_ID,SampleID,ASM,SameSample,Illumina_ID
4,T0206,S0206-01,MFS-635,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBi...,1,MFS-635


In [155]:
df_personal_ASM_for_aln.query("pid=='T0082'")

,pid,SampleID,ASM,SameSample


In [125]:
len(df_personal_ASM_for_aln), df_personal_ASM_for_aln.ASM.nunique()

(179, 154)

In [120]:
set(os.listdir(personal_ref_dir)) - set(df_personal_ASM_for_aln.SampleID)

set()

In [121]:
set(df_personal_ASM_for_aln.SampleID) - set(os.listdir(personal_ref_dir))

set()

In [147]:
df_personal_ASM_for_aln.query("ASM.str.contains('S0082')")

,pid,SampleID,ASM,SameSample


In [144]:
# shutil.rmtree(f"{personal_ref_dir}/")

'/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_personal_assembly'

In [122]:
len(df_hybridASM_QC), df_hybridASM_QC.Original_ID.nunique()

(190, 186)

In [283]:
df_send = df_hybridASM_QC.query("numContigs==1 & circContigLength >= 4000000 & Flye_Coll2014 == Flye_PP_Coll2014").reset_index(drop=True)

df_send.iloc[df_send.index.values[df_send.duplicated(subset='Original_ID', keep=False)]]

# .drop_duplicates(subset='Original_ID')

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014
26,S0077-06,1,4421214,4.1.2.1,4.1.2.1,0,MFS-44,MFS-44,0.021498,4.1.2.1
27,S0077-06,1,4421214,4.1.2.1,4.1.2.1,0,MFS-213,MFS-44,0.021498,4.1.2.1
80,S0227-07,1,4404783,4.1.1.1,4.1.1.1,48,MFS-241,MFS-136,0.007964,4.1.1.1
81,S0227-07,1,4404783,4.1.1.1,4.1.1.1,48,MFS-136,MFS-136,0.007964,4.1.1.1


In [300]:
df_run = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBio_Illumina_sample_mapping.csv")

In [317]:
df_dups_run = df_run.iloc[df_run.index.values[df_run.duplicated(subset='Original_ID', keep=False)]].query("PacBio_fastlin==Illumina_Coll2014")

samples_with_single_sample = df_dups_run.drop_duplicates(subset='Original_ID', keep=False).Original_ID.unique()

df_dups_run.query("Original_ID not in @samples_with_single_sample")[['PacBio_ID', 'PacBio_FQ_PATH', 'Illumina_ID']].rename(columns={'PacBio_ID': 'Original_ID'})#.to_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/data/LR_SR_combined_data_to_run.tsv", sep='\t', index=False)

,Original_ID,PacBio_FQ_PATH,Illumina_ID
34,MFS-44,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,MFS-44
35,MFS-213,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,MFS-44
36,MFS-214,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,MFS-47
37,MFS-47,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,MFS-47
106,MFS-241,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,MFS-136
107,MFS-136,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,MFS-136


In [318]:
df_run.iloc[df_run.index.values[df_run.duplicated(subset='Original_ID', keep=False)]]

,Original_ID,PacBio_ID,Dataset_Tag,PacBio_FQ_PATH,PacBio_fastlin,Illumina_ID,Illumina_Kraken_Unclassified_Percent,Illumina_F2,Illumina_Coll2014
32,S0077-01,MFS-43,TRUST_PB_Set1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.2.1,MFS-43,0.47,0.021399,4.1.2.1
33,S0077-01,MFS-212,TRUST_PB_Set2,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.1.3,MFS-43,0.47,0.021399,4.1.2.1
34,S0077-06,MFS-44,TRUST_PB_Set1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.2.1,MFS-44,0.66,0.021498,4.1.2.1
35,S0077-06,MFS-213,TRUST_PB_Set4,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.2.1,MFS-44,0.66,0.021498,4.1.2.1
36,S0082-01,MFS-214,TRUST_PB_Set2,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,3.1.1,MFS-47,0.62,0.012186,3.1.1
37,S0082-01,MFS-47,TRUST_PB_Set1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,3.1.1,MFS-47,0.62,0.012186,3.1.1
106,S0227-07,MFS-241,TRUST_PB_Set3,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.1.1,MFS-136,0.69,0.007964,4.1.1.1
107,S0227-07,MFS-136,TRUST_PB_Set1,/n/data1/hms/dbmi/farhat/fastq_db/TRUST_PacBio...,4.1.1.1,MFS-136,0.69,0.007964,4.1.1.1


In [312]:
df_hybridASM_QC.query("Original_ID=='S0077-01'")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014
32,S0077-01,2,NaN,4.1.2.1,4.1.2.1,1,MFS-43,MFS-43,0.021399,4.1.2.1
33,S0077-01,2,NaN,4.1.2.1,4.1.2.1,1,MFS-212,MFS-43,0.021399,4.1.2.1


In [293]:
df_hybridASM_QC.query("Original_ID in ['S0077-06', 'S0227-07']")

,Original_ID,numContigs,circContigLength,Flye_Coll2014,Flye_PP_Coll2014,NumPilonChanges,PacBio_ID,Illumina_ID,Illumina_F2,Illumina_Coll2014
34,S0077-06,1,4421214,4.1.2.1,4.1.2.1,0,MFS-44,MFS-44,0.021498,4.1.2.1
35,S0077-06,1,4421214,4.1.2.1,4.1.2.1,0,MFS-213,MFS-44,0.021498,4.1.2.1
106,S0227-07,1,4404783,4.1.1.1,4.1.1.1,48,MFS-241,MFS-136,0.007964,4.1.1.1
107,S0227-07,1,4404783,4.1.1.1,4.1.1.1,48,MFS-136,MFS-136,0.007964,4.1.1.1
